In [6]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
from pathlib import Path

load_dotenv()

client = Anthropic(
    default_headers={
        "anthropic-beta": "code-execution-2025-08-25, files-api-2025-04-14"
    }
)
model = "claude-sonnet-4-5-20250929"

In [12]:
# Helper functions
import base64
from anthropic.types import Message
from IPython.display import display, HTML, Image as IPImage


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


class PrettyMessage:
    """Wraps a Claude Message for pretty Jupyter display."""

    def __init__(self, message):
        self._message = message

    def __getattr__(self, name):
        return getattr(self._message, name)

    def _repr_html_(self):
        parts = []
        for block in self._message.content:
            btype = block.type

            if btype == "text":
                import html
                text = html.escape(block.text)
                text = text.replace("\n", "<br>")
                parts.append(f'<div style="margin:8px 0">{text}</div>')

            elif btype == "thinking":
                import html
                text = html.escape(block.thinking).replace("\n", "<br>")
                parts.append(
                    f'<details style="margin:8px 0;border:1px solid #e0c97a;border-radius:6px;padding:6px">'
                    f'<summary style="cursor:pointer;font-weight:bold;color:#b8860b">Thinking</summary>'
                    f'<div style="margin-top:6px;color:#555">{text}</div></details>'
                )

            elif btype == "tool_use":
                import html, json
                inp = html.escape(json.dumps(block.input, indent=2))
                parts.append(
                    f'<details style="margin:8px 0;border:1px solid #7ab8e0;border-radius:6px;padding:6px">'
                    f'<summary style="cursor:pointer;font-weight:bold;color:#1a6a9a">Tool use: <code>{block.name}</code></summary>'
                    f'<pre style="margin-top:6px;background:#f4f8fc;padding:8px;border-radius:4px;overflow:auto">{inp}</pre>'
                    f'</details>'
                )

            elif btype == "server_tool_use":
                import html, json
                inp = html.escape(json.dumps(block.input, indent=2))
                parts.append(
                    f'<details style="margin:8px 0;border:1px solid #7ab8e0;border-radius:6px;padding:6px">'
                    f'<summary style="cursor:pointer;font-weight:bold;color:#1a6a9a">Code execution input</summary>'
                    f'<pre style="margin-top:6px;background:#f4f8fc;padding:8px;border-radius:4px;overflow:auto">{inp}</pre>'
                    f'</details>'
                )

            elif btype == "server_tool_result" or btype == "tool_result":
                result_blocks = getattr(block, "content", [])
                inner_parts = []
                for rb in result_blocks:
                    if getattr(rb, "type", None) == "text":
                        import html
                        t = html.escape(rb.text).replace("\n", "<br>")
                        inner_parts.append(f'<div style="font-family:monospace;font-size:0.85em">{t}</div>')
                    elif getattr(rb, "type", None) == "image":
                        img_data = rb.source.data
                        media = rb.source.media_type
                        inner_parts.append(
                            f'<img src="data:{media};base64,{img_data}" '
                            f'style="max-width:100%;border-radius:4px;margin-top:6px">'
                        )
                label = "Code execution result"
                parts.append(
                    f'<details open style="margin:8px 0;border:1px solid #90c090;border-radius:6px;padding:6px">'
                    f'<summary style="cursor:pointer;font-weight:bold;color:#2a7a2a">{label}</summary>'
                    f'<div style="margin-top:6px">{"".join(inner_parts)}</div></details>'
                )

        meta = (
            f'<div style="font-size:0.8em;color:#999;margin-top:10px">'
            f'model={self._message.model} | '
            f'stop={self._message.stop_reason} | '
            f'in={self._message.usage.input_tokens} out={self._message.usage.output_tokens} tokens'
            f'</div>'
        )
        return (
            f'<div style="font-family:sans-serif;line-height:1.5;'
            f'border:1px solid #ddd;border-radius:8px;padding:12px;background:#fafafa">'
            + "".join(parts) + meta +
            f'</div>'
        )


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=2000,
):
    params = {
        "model": model,
        "max_tokens": 10000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])


def upload(file_path):
    path = Path(file_path)
    extension = path.suffix.lower()

    mime_type_map = {
        ".pdf": "application/pdf",
        ".txt": "text/plain",
        ".md": "text/plain",
        ".py": "text/plain",
        ".js": "text/plain",
        ".html": "text/plain",
        ".css": "text/plain",
        ".csv": "text/csv",
        ".json": "application/json",
        ".xml": "application/xml",
        ".xlsx": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        ".xls": "application/vnd.ms-excel",
        ".jpeg": "image/jpeg",
        ".jpg": "image/jpeg",
        ".png": "image/png",
        ".gif": "image/gif",
        ".webp": "image/webp",
    }

    mime_type = mime_type_map.get(extension)

    if not mime_type:
        raise ValueError(f"Unknown mimetype for extension: {extension}")
    filename = path.name

    with open(file_path, "rb") as file:
        return client.beta.files.upload(file=(filename, file, mime_type))


def list_files():
    return client.beta.files.list()


def delete_file(id):
    return client.beta.files.delete(id)


def download_file(id, filename=None):
    file_content = client.beta.files.download(id)

    if not filename:
        file_metadata = get_metadata(id)
        file_content.write_to_file(file_metadata.filename)
    else:
        file_content.write_to_file(filename)


def get_metadata(id):
    return client.beta.files.retrieve_metadata(id)

In [8]:
file_metadata = upload("streaming.csv")
file_metadata

FileMetadata(id='file_011CZi9mnCDLdRgjmuoZnSTR', created_at=datetime.datetime(2026, 4, 4, 5, 8, 3, 200000, tzinfo=datetime.timezone.utc), filename='streaming.csv', mime_type='text/csv', size_bytes=25733, type='file', downloadable=False)

In [15]:
messages = []

add_user_message(
    messages,
    [
        {
            "type": "text",
            "text": """
Run a detailed analysis to determine major drivers of churn.
Your final output should include at least one detailed plot summarizing your findings.

Critical note: Every time you execute code, you're starting with a completely clean slate. 
No variables or library imports from previous executions exist. You need to redeclare/reimport all variables/libraries.
            """,
        },
        {"type": "container_upload", "file_id": file_metadata.id},
    ],
)

chat(messages, tools=[{"type": "code_execution_20250825", "name": "code_execution"}])

Message(id='msg_01F5jvE47jkzLMuwBJYGLgaJ', container=Container(id='container_011CZiBQEGyJRrmrZyunaeco', expires_at=datetime.datetime(2026, 4, 4, 6, 36, 26, 408794, tzinfo=TzInfo(0))), content=[TextBlock(citations=None, text="I'll analyze the streaming.csv file to determine the major drivers of churn. Let me start by exploring the data and then perform a comprehensive analysis.", type='text'), ServerToolUseBlock(id='srvtoolu_01RCRTpR2V2AeHYSnQJ3AzBL', caller=DirectCaller(type='direct'), input={'command': 'cd $INPUT_DIR && head -20 streaming.csv'}, name='bash_code_execution', type='server_tool_use'), BashCodeExecutionToolResultBlock(content=BashCodeExecutionResultBlock(content=[], return_code=0, stderr='', stdout='UserID,SubscriptionTier,TotalViewingHoursLastMonth,TopGenre,BingeWatchingSessionsLastMonth,NumberOfUniqueTitlesWatchedLastMonth,AverageSessionDurationMinutes,CustomerServiceInteractionsLastYear,MonthlyCost,Churned\nUSER_00001,Basic,47.9,Comedy,5,15,32.6,3,7.99,0\nUSER_00002,Pre

In [17]:
download_file("file_011CZiBwFfBUeXjBriNaH4Mr")